In [1]:
from flask import Flask, request, render_template_string
import numpy as np
import joblib
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_input_vgg
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_input_inception
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_input_resnet
import matplotlib.pyplot as plt
import io
import base64
from tensorflow.keras.layers import LSTM, Bidirectional


In [2]:
stacking_clf = joblib.load('MITBIHDATA/MODELS/MIT_stacking_clf.pkl')

model_resnet = load_model('MITBIHDATA/MODELS/resnet_model.h5')

model_vgg16 = load_model('MITBIHDATA/MODELS/vgg16_model.h5')

model_inception = load_model('MITBIHDATA/MODELS/inception_model.h5')

C:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.2.2 when using version 1.3.0. This might lead to breaking code

In [3]:
X_test_resized = np.load('MITBIHDATA/TEST2/X_test_resized.npy')
y_test = np.load('MITBIHDATA/TEST2/y_test.npy')
X_test_meta = np.load('MITBIHDATA/TEST2/X_test_meta.npy')

In [4]:
X_test_resized_vgg = preprocess_input_vgg(np.repeat(X_test_resized, 3, axis=-1))
X_test_resized_inception = preprocess_input_inception(np.repeat(X_test_resized, 3, axis=-1))
X_test_resized_resnet = preprocess_input_resnet(np.repeat(X_test_resized, 3, axis=-1))

In [5]:
model_cnnlstm = load_model('MITBIHDATA/MODELS/CNN_lstm.keras')
model_cnnbilstm = load_model('MITBIHDATA/MODELS/CNNBILSTM.keras')
model_cnn = load_model('MITBIHDATA/MODELS/CNN.keras')
stacking_clf2 = joblib.load('MITBIHDATA/MODELS/ensemble.pkl')

In [6]:
X_test = np.load('MITBIHDATA/TEST1/X_test.npy')
y_test2 = np.load('MITBIHDATA/TEST1/y_test.npy')
X_test_meta2 = np.load('MITBIHDATA/TEST1/X_train_meta.npy')

In [7]:
PHYSIO_cnnlstm = load_model('PHYSIODATA/MODELS/CNN_LTSM10.h5')
PHYSIO_cnnbilstm = load_model('PHYSIODATA/MODELS/CNNbi10.h5')
PHYSIO_cnn = load_model('PHYSIODATA/MODELS/CNN10.h5')
PHYSIO_stacking_clf = joblib.load('PHYSIODATA/MODELS/stacking_ensemble_model.pkl')

In [8]:
X_test_PHYSIO = np.load('PHYSIODATA/TEST1/X_test.npy')
y_test_PHYSIO = np.load('PHYSIODATA/TEST1/y_test.npy')
X_test_meta_PHYSIO = np.load('PHYSIODATA/TEST1/X_test_meta_.npy')

In [9]:
PHYSIO_ensemble = joblib.load('PHYSIODATA/MODELS/PHYSIO_stacking_clf.pkl')

PHYSIO_resnet = load_model('PHYSIODATA/MODELS/PHYSIOresnet.h5')

PHYSIO_vgg16 = load_model('PHYSIODATA/MODELS/PHYSIOvgg.h5')

PHYSIO_inception = load_model('PHYSIODATA/MODELS/PHYSIOinception.h5')

In [10]:
X_test_PHYSIO_res = np.load('PHYSIODATA/TEST2/X_test_resized.npy')
y_test_PHYSIO_res = np.load('PHYSIODATA/TEST2/y_test_resized.npy')
X_test_meta_PHYSIO_res = np.load('PHYSIODATA/TEST2/X_test_meta_PHYSIO.npy')

In [11]:
X_test_res_vgg = preprocess_input_vgg(np.repeat(X_test_PHYSIO_res, 3, axis=-1))
X_test_res_inception = preprocess_input_inception(np.repeat(X_test_PHYSIO_res, 3, axis=-1))
X_test_res_resnet = preprocess_input_resnet(np.repeat(X_test_PHYSIO_res, 3, axis=-1))

In [12]:
app = Flask(__name__)

In [13]:
@app.route('/')
def index():
    return render_template_string('''
        <!DOCTYPE html>
        <html lang="en">
        <head>
            <meta charset="UTF-8">
            <meta name="viewport" content="width=device-width, initial-scale=1.0">
            <title>ECG Signal Prediction</title>
            <style>
                body {
                    font-family: Arial, sans-serif;
                    background-color: #f8f9fa;
                    color: #333;
                    margin: 0;
                    padding: 0;
                }
                header {
                    background-color: #007bff;
                    color: #fff;
                    padding: 20px;
                    text-align: center;
                    box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
                }
                h1 {
                    margin: 0;
                    font-size: 24px;
                }
                main {
                    padding: 20px;
                }
                .tab-buttons,
                .nested-tab-buttons {
                    display: flex;
                    justify-content: center;
                    margin-bottom: 20px;
                }
                button {
                    padding: 10px 20px;
                    margin: 0 5px;
                    cursor: pointer;
                    border: none;
                    border-radius: 5px;
                    background-color: #007bff;
                    color: #fff;
                    font-size: 16px;
                    transition: background-color 0.3s ease;
                }
                button:hover {
                    background-color: #0056b3;
                }
                .tab,
                .nested-tab {
                    display: none;
                }
                .tab.active,
                .nested-tab.active {
                    display: block;
                }
                form {
                    max-width: 400px;
                    margin: 0 auto;
                    padding: 20px;
                    border: 1px solid #ddd;
                    border-radius: 5px;
                    background-color: #fff;
                    box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
                }
                label {
                    display: block;
                    margin-bottom: 8px;
                    font-weight: bold;
                }
                input[type="number"],
                select {
                    width: 100%;
                    padding: 10px;
                    margin-bottom: 20px;
                    border: 1px solid #ddd;
                    border-radius: 5px;
                }
                button[type="submit"] {
                    background-color: #28a745;
                }
                button[type="submit"]:hover {
                    background-color: #218838;
                }
            </style>
        </head>
        <body>
            <header>
                <h1>ECG Signal Prediction</h1>
            </header>
            <main>
                <div class="tab-buttons">
                    <button type="button" onclick="showTab('tab-dataset1')" class="active">Dataset 1</button>
                    <button type="button" onclick="showTab('tab-dataset2')">Dataset 2</button>
                </div>

                <div id="tab-dataset1" class="tab active">
                    <div class="nested-tab-buttons">
                        <button type="button" onclick="showNestedTab('nested-tab-1d-dataset1')" class="active">1D Models</button>
                        <button type="button" onclick="showNestedTab('nested-tab-2d-dataset1')">2D Models</button>
                    </div>
                    <div id="nested-tab-1d-dataset1" class="nested-tab active">
                        <form action="/predict" method="post">
                            <input type="hidden" name="dataset" value="dataset1">
                            <input type="hidden" name="model_type" value="1d">
                            <label for="index">Enter Index:</label>
                            <input type="number" id="index" name="index" required>
                            <label for="model">Choose Model:</label>
                            <select id="model" name="model">
                                <option value="cnn">CNN</option>
                                <option value="cnnbilstm">CNN BiLSTM</option>
                                <option value="cnnlstm">CNN LSTM</option>
                                <option value="stacking_clf2">Stacking CLF 2</option>
                            </select>
                            <button type="submit">Predict</button>
                        </form>
                    </div>
                    <div id="nested-tab-2d-dataset1" class="nested-tab">
                        <form action="/predict" method="post">
                            <input type="hidden" name="dataset" value="dataset1">
                            <input type="hidden" name="model_type" value="2d">
                            <label for="index">Enter Index:</label>
                            <input type="number" id="index" name="index" required>
                            <label for="model">Choose Model:</label>
                            <select id="model" name="model">
                                <option value="vgg16">VGG16</option>
                                <option value="inception">InceptionV3</option>
                                <option value="resnet">ResNet50</option>
                                <option value="ensemble">Ensemble</option>
                            </select>
                            <button type="submit">Predict</button>
                        </form>
                    </div>
                </div>

                <div id="tab-dataset2" class="tab">
                    <div class="nested-tab-buttons">
                        <button type="button" onclick="showNestedTab('nested-tab-1d-dataset2')" class="active">1D Models</button>
                        <button type="button" onclick="showNestedTab('nested-tab-2d-dataset2')">2D Models</button>
                    </div>
                    <div id="nested-tab-1d-dataset2" class="nested-tab active">
                        <form action="/predict" method="post">
                            <input type="hidden" name="dataset" value="dataset2">
                            <input type="hidden" name="model_type" value="1d">
                            <label for="index">Enter Index:</label>
                            <input type="number" id="index" name="index" required>
                            <label for="model">Choose Model:</label>
                            <select id="model" name="model">
                                <option value="PHYSIO_cnn">PHYSIO CNN</option>
                                <option value="PHYSIO_cnnbilstm">PHYSIO CNN BiLSTM</option>
                                <option value="PHYSIO_cnnlstm">PHYSIO CNN LSTM</option>
                                <option value="PHYSIO_stacking_clf">PHYSIO Stacking CLF</option>
                            </select>
                            <button type="submit">Predict</button>
                        </form>
                    </div>
                    <div id="nested-tab-2d-dataset2" class="nested-tab">
                        <form action="/predict" method="post">
                            <input type="hidden" name="dataset" value="dataset2">
                            <input type="hidden" name="model_type" value="2d">
                            <label for="index">Enter Index:</label>
                            <input type="number" id="index" name="index" required>
                            <label for="model">Choose Model:</label>
                            <select id="model" name="model">
                                <option value="PHYSIO_vgg16">PHYSIO VGG16</option>
                                <option value="PHYSIO_inception">PHYSIO InceptionV3</option>
                                <option value="PHYSIO_resnet">PHYSIO ResNet50</option>
                                <option value="PHYSIO_ensemble">PHYSIO Ensemble</option>
                            </select>
                            <button type="submit">Predict</button>
                        </form>
                    </div>
                </div>
            </main>
            <script>
                function showTab(tabId) {
                    var tabs = document.querySelectorAll('.tab');
                    tabs.forEach(function(tab) {
                        tab.classList.remove('active');
                    });
                    document.getElementById(tabId).classList.add('active');
                }

                function showNestedTab(nestedTabId) {
                    var nestedTabs = document.querySelectorAll('.nested-tab');
                    nestedTabs.forEach(function(nestedTab) {
                        nestedTab.classList.remove('active');
                    });
                    document.getElementById(nestedTabId).classList.add('active');
                }
            </script>
        </body>
        </html>
    ''')


In [14]:
label_mapping = {0: 'Normal', 1: 'Left bundle branch block', 2: 'Right bundle branch block', 3: 'Atrial premature', 4: 'Premature ventricular contraction'}
label_mapping_physio = {0: 'AF', 1: 'Normal', 2: 'Other rhythm', 3: 'Noise'}
@app.route('/predict', methods=['POST'])
def predict():
    index = int(request.form['index'])
    model_choice = request.form['model']
    model_type = request.form['model_type']
    dataset = request.form['dataset']

    if dataset == 'dataset1':
        if model_type == '2d':
            if model_choice == 'vgg16':
                model = model_vgg16
                X_test_resized_model = X_test_resized_vgg
            elif model_choice == 'inception':
                model = model_inception
                X_test_resized_model = X_test_resized_inception
            elif model_choice == 'resnet':
                model = model_resnet
                X_test_resized_model = X_test_resized_resnet
            elif model_choice == 'ensemble':
                y_pred = stacking_clf.predict(X_test_meta)
                true_label = label_mapping[y_test[index]]
                predicted_label = label_mapping[y_pred[index]]
                plt.figure(figsize=(10, 4))
                plt.plot(X_test_resized[index].flatten())
            else:
                return "Invalid model choice"

            if model_choice != 'ensemble':
                y_pred_probs = model.predict(X_test_resized_model[index:index+1])
                y_pred = np.argmax(y_pred_probs, axis=1)
                true_label = label_mapping[y_test[index]]
                predicted_label = label_mapping[y_pred[0]]

                plt.figure(figsize=(10, 4))
                plt.plot(X_test_resized[index].flatten())

        elif model_type == '1d':
            if model_choice == 'cnn':
                model = model_cnn
                X_test_resized_model = X_test
            elif model_choice == 'cnnbilstm':
                model = model_cnnbilstm
                X_test_resized_model = X_test
            elif model_choice == 'cnnlstm':
                model = model_cnnlstm
                X_test_resized_model = X_test
            elif model_choice == 'stacking_clf2':
                y_pred = stacking_clf2.predict(X_test_meta2)
                true_label = label_mapping[y_test2[index]]
                predicted_label = label_mapping[y_pred[index]]
                plt.figure(figsize=(10, 4))
                plt.plot(X_test[index])
            else:
                return "Invalid model choice"

            if model_choice != 'stacking_clf2':
                y_pred_probs = model.predict(X_test_resized_model[index:index+1])
                y_pred = np.argmax(y_pred_probs, axis=1)
                true_label = label_mapping[y_test2[index]]
                predicted_label = label_mapping[y_pred[0]]

                plt.figure(figsize=(10, 4))
                plt.plot(X_test[index])

    elif dataset == 'dataset2':
        if model_type == '1d':
            if model_choice == 'PHYSIO_cnn':
                model = PHYSIO_cnn
                X_test_resized_model = X_test_PHYSIO
            elif model_choice == 'PHYSIO_cnnbilstm':
                model = PHYSIO_cnnbilstm
                X_test_resized_model = X_test_PHYSIO
            elif model_choice == 'PHYSIO_cnnlstm':
                model = PHYSIO_cnnlstm
                X_test_resized_model = X_test_PHYSIO
            elif model_choice == 'PHYSIO_stacking_clf':
                y_pred = PHYSIO_stacking_clf.predict(X_test_meta_PHYSIO)
                true_label = label_mapping_physio[y_test_PHYSIO[index]]
                predicted_label = label_mapping_physio[y_pred[index]]
                plt.figure(figsize=(10, 4))
                plt.plot(X_test_PHYSIO[index])
            else:
                return "Invalid model choice"

            if model_choice != 'PHYSIO_stacking_clf':
                y_pred_probs = model.predict(X_test_resized_model[index:index+1])
                y_pred = np.argmax(y_pred_probs, axis=1)
                true_label = label_mapping_physio[y_test_PHYSIO[index]]
                predicted_label = label_mapping_physio[y_pred[0]]

                plt.figure(figsize=(10, 4))
                plt.plot(X_test_PHYSIO[index])
        
        elif model_type == '2d':
            if model_choice == 'PHYSIO_vgg16':
                model = PHYSIO_vgg16
                X_test_resized_model = X_test_res_vgg
            elif model_choice == 'PHYSIO_inception':
                model = PHYSIO_inception
                X_test_resized_model = X_test_res_inception
            elif model_choice == 'PHYSIO_resnet':
                model = PHYSIO_resnet
                X_test_resized_model = X_test_res_resnet
            elif model_choice == 'PHYSIO_ensemble':
                y_pred = PHYSIO_ensemble.predict(X_test_meta_PHYSIO_res)
                true_label = label_mapping_physio[y_test_PHYSIO_res[index]]
                predicted_label = label_mapping_physio[y_pred[index]]
                plt.figure(figsize=(10, 4))
                plt.plot(X_test_PHYSIO_res[index].flatten())
            else:
                return "Invalid model choice"

            if model_choice != 'PHYSIO_ensemble':
                y_pred_probs = model.predict(X_test_resized_model[index:index+1])
                y_pred = np.argmax(y_pred_probs, axis=1)
                true_label = label_mapping_physio[y_test_PHYSIO_res[index]]
                predicted_label = label_mapping_physio[y_pred[0]]

                plt.figure(figsize=(10, 4))
                plt.plot(X_test_PHYSIO_res[index].flatten())

    plt.title(f'True Label: {true_label}, Predicted Label: {predicted_label}')
    plt.xlabel('Sample')
    plt.ylabel('Amplitude')

    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    img_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    plt.close()

    return render_template_string('''
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prediction Result</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                background-color: #f8f9fa;
                color: #333;
                margin: 0;
                padding: 0;
            }
            header {
                background-color: #007bff;
                color: #fff;
                padding: 20px;
                text-align: center;
                box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
            }
            h1 {
                margin: 0;
                font-size: 24px;
            }
            main {
                padding: 20px;
            }
            p {
                margin-bottom: 10px;
            }
            img {
                margin-top: 20px;
                display: block;
                max-width: 100%;
                height: auto;
                border: 1px solid #ddd;
                border-radius: 5px;
                box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
            }
            a {
                display: inline-block;
                margin-top: 20px;
                padding: 10px 20px;
                background-color: #007bff;
                color: #fff;
                text-decoration: none;
                border-radius: 5px;
                box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
                transition: background-color 0.3s ease;
            }
            a:hover {
                background-color: #0056b3;
            }
            form {
                max-width: 400px;
                margin: 0 auto;
                padding: 20px;
                border: 1px solid #ddd;
                border-radius: 5px;
                background-color: #fff;
                box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
            }
            label {
                display: block;
                margin-bottom: 8px;
                font-weight: bold;
            }
            input[type="number"],
            select {
                width: 100%;
                padding: 10px;
                margin-bottom: 20px;
                border: 1px solid #ddd;
                border-radius: 5px;
            }
            button[type="submit"] {
                background-color: #28a745;
            }
            button[type="submit"]:hover {
                background-color: #218838;
            }
        </style>
    </head>
    <body>
        <header>
            <h1>Prediction Result</h1>
        </header>
        <main>
            <p>Index: {{ index }}</p>
            <p>Model: {{ model_choice }}</p>
            <p>True Label: {{ true_label }}</p>
            <p>Predicted Label: {{ predicted_label }}</p>
            <img src="data:image/png;base64,{{ img_base64 }}" alt="ECG Signal">
            <br>
            <form action="/predict" method="post">
                <input type="hidden" name="dataset" value="{{ dataset }}">
                <input type="hidden" name="model_type" value="{{ model_type }}">
                <label for="index">Enter Index:</label>
                <input type="number" id="index" name="index" required>
                <label for="model">Choose Model:</label>
                <select id="model" name="model">
                    {% if dataset == 'dataset1' %}
                        {% if model_type == '2d' %}
                            <option value="vgg16">VGG16</option>
                            <option value="inception">InceptionV3</option>
                            <option value="resnet">ResNet50</option>
                            <option value="ensemble">Ensemble</option>
                        {% else %}
                            <option value="cnn">CNN</option>
                            <option value="cnnbilstm">CNN BiLSTM</option>
                            <option value="cnnlstm">CNN LSTM</option>
                            <option value="stacking_clf2">Stacking CLF 2</option>
                        {% endif %}
                    {% else %}
                        {% if model_type == '2d' %}
                            <option value="PHYSIO_vgg16">PHYSIO VGG16</option>
                            <option value="PHYSIO_inception">PHYSIO InceptionV3</option>
                            <option value="PHYSIO_resnet">PHYSIO ResNet50</option>
                            <option value="PHYSIO_ensemble">PHYSIO Ensemble</option>
                        {% else %}
                            <option value="PHYSIO_cnn">PHYSIO CNN</option>
                            <option value="PHYSIO_cnnbilstm">PHYSIO CNN BiLSTM</option>
                            <option value="PHYSIO_cnnlstm">PHYSIO CNN LSTM</option>
                            <option value="PHYSIO_stacking_clf">PHYSIO Stacking CLF</option>
                        {% endif %}
                    {% endif %}
                </select>
                <button type="submit">Predict</button>
            </form>
            <br>
            <a href="/">Back to Home</a>
        </main>
    </body>
    </html>
''', true_label=true_label, predicted_label=predicted_label, img_base64=img_base64, index=index, model_choice=model_choice, dataset=dataset, model_type=model_type)


In [ ]:
# Run the Flask app
if __name__ == '__main__':
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [08/Jun/2024 05:01:52] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Jun/2024 05:01:52] "GET /favicon.ico HTTP/1.1" 404 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 519ms/step


INFO:werkzeug:127.0.0.1 - - [08/Jun/2024 05:02:55] "POST /predict HTTP/1.1" 200 -
